# 03. Logprobs for LLM confidence

This notebook shows how token logprobs can be used as a confidence signal, while being careful not to confuse confidence with truth.

## Important caution

Logprobs reflect **model confidence under its own distribution**, not truth probability.

A model can be confident and wrong.

In [ ]:

import math
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "sshleifer/tiny-gpt2"
device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()


In [ ]:

def score_continuation(prompt, continuation):
    full_text = prompt + continuation
    inputs = tokenizer(full_text, return_tensors='pt')
    prompt_inputs = tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    logprobs = torch.log_softmax(outputs.logits[0], dim=-1)
    target_ids = inputs['input_ids'][0]
    prompt_len = prompt_inputs['input_ids'].shape[1]
    token_logprobs = []
    for pos in range(prompt_len - 1, len(target_ids) - 1):
        next_id = target_ids[pos + 1]
        token_logprobs.append(logprobs[pos, next_id].item())
    return token_logprobs


In [ ]:

prompt = 'Question: What color is the sky on a clear day?
Answer:'
continuation = ' Blue.'
logprobs = score_continuation(prompt, continuation)
print('token logprobs:', logprobs)
print('average token logprob:', sum(logprobs)/len(logprobs))


## Simple thresholding logic

A workflow might flag generations with very low average logprob for extra review.

In [ ]:

avg_lp = sum(logprobs) / len(logprobs)
threshold = -5.0
print('flag for review?', avg_lp < threshold)


## Example with retrieved context

In a RAG-style workflow, extra context can shift the model distribution and change logprobs.

In [ ]:

base_prompt = 'Question: Who wrote Pride and Prejudice?
Answer:'
rag_prompt = 'Context: Pride and Prejudice is a novel by Jane Austen.

Question: Who wrote Pride and Prejudice?
Answer:'
continuation = ' Jane Austen.'

lp_no_rag = score_continuation(base_prompt, continuation)
lp_with_rag = score_continuation(rag_prompt, continuation)

print('avg logprob without retrieved context:', sum(lp_no_rag)/len(lp_no_rag))
print('avg logprob with retrieved context:', sum(lp_with_rag)/len(lp_with_rag))
